# `validation.ipynb`: check that models converge to biologically reasonable stable states

In [1]:
from jax import numpy as jnp
from scipy.stats import pearsonr

from utils.stability import *
from utils.modelIO import *

In [2]:
GRN_PATH = 'data/eml/eml_fate_network.npy'
GIDS_PATH = 'data/eml/eml_gene_ids.csv'
SIM_PATH = 'results/eml/bio_T_coarse/bio_cT_00_xs.npy'

grn, gids, N = load_grn(GRN_PATH, GIDS_PATH)
xs = jnp.load(SIM_PATH)

In [3]:
stable_states = get_stable_states(grn, jnp.zeros(N))
final_states = xs[:, -1]
print('All final states are stable: {}'.format(jnp.all(jnp.isin(final_states, stable_states))))

All final states are stable: True


In [4]:
def get_expression_level_instance(grn_states):
    return get_expression_level(grn_states, gids, N)
get_expression_level_optimized = jax.jit(jax.vmap(get_expression_level_instance))
elevels = get_expression_level_optimized(xs)

In [5]:
hi_PU1 = elevels['PU.1'] > 0.95
lo_PU1 = elevels['PU.1'] < 0.05

hi_GATA1 = elevels['GATA1'] > 0.95
lo_GATA1 = elevels['GATA1'] < 0.05

hi_cKit = elevels['c-Kit'] > 0.95
lo_cKit = elevels['c-Kit'] < 0.05

hi_Sca1 = elevels['Sca1'] > 0.95
lo_Sca1 = elevels['Sca1'] < 0.05

In [6]:
erythroid = jnp.sum(hi_GATA1*lo_PU1)
myeloid = jnp.sum(lo_GATA1*hi_PU1)
both_lo = jnp.sum(lo_GATA1*lo_PU1)
both_hi = jnp.sum(hi_GATA1*hi_PU1)

In [7]:
print('Number of erythroid: {}'.format(erythroid))
print('Number of myeloid: {}'.format(myeloid))
print('Number expressing neither GATA1 nor PU.1: {}'.format(both_lo))
print('Number expressing both GATA1 and PU.1: {}'.format(both_hi))

Number of erythroid: 382
Number of myeloid: 376
Number expressing neither GATA1 nor PU.1: 124
Number expressing both GATA1 and PU.1: 118


In [8]:
erythroid_Sca1_plus = jnp.sum(hi_GATA1*lo_PU1*hi_Sca1)
myeloid_Sca1_plus = jnp.sum(lo_GATA1*hi_PU1*hi_Sca1)
print('Sca1+ erythroid: {}'.format(erythroid_Sca1_plus))
print('Sca1+ myeloid: {}'.format(myeloid_Sca1_plus))

Sca1+ erythroid: 382
Sca1+ myeloid: 0


In [9]:
pearsonr(elevels['GATA1'], elevels['PU.1'])

PearsonRResult(statistic=np.float64(-0.5160371479578316), pvalue=np.float32(0.0))